# Track B — Fase 2: Probe Grid (backbone × head)

**BDC Satria Data 2026** | Cari kombinasi embedding × classifier terbaik dari OOF.

Ini tahap yang **akhirnya menghasilkan angka akurasi**. `extract_all` cuma mengubah
gambar → vektor; di sini vektor itu dilatih jadi classifier 5-fold dan diukur Macro-F1-nya.

**Yang diuji:**
- 5 backbone tunggal × 4 head (linear, MLP, LightGBM, kNN) = 20 kombinasi
- 3 kombinasi concat multi-backbone × 2 head = 6 kombinasi
- Total 26 — semua di CPU, ~10–30 menit

**Cara baca pemenang (JANGAN asal ambil `mean` tertinggi):**
1. `mean` tinggi TAPI `std` antar-fold wajar (tidak jauh lebih besar dari lain)
2. Kalau dua kandidat `mean`-nya beda < 0.002 → pilih yang `min` (fold terburuk) lebih tinggi
3. `mean` naik tapi `min` turun → **TOLAK** (mengejar satu fold beruntung, bukan gain nyata)

> Semua di CPU — **tidak butuh GPU**. Bisa jalan walau kuota T4 habis.

---
## 🔧 SETUP

In [ ]:
# Cell 1 — Repo + Drive + dependensi (CPU, tanpa GPU)
import os
if not os.path.exists('/content/satria-data-bdcugm02'):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git
else:
    !git -C /content/satria-data-bdcugm02 pull

!pip install -q scikit-learn lightgbm

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ setup siap')

In [ ]:
# Cell 2 — Verifikasi 20 embedding ADA sebelum grid (biar tak gagal di tengah)
import os, numpy as np
EMB = "/content/drive/MyDrive/BDC2026apace/output_trackB/embeddings"
NAMES = ['siglip2b256', 'siglip2so400m', 'siglip1b256', 'dinov3vitl', 'dinov3cnxb']

ok = True
for n in NAMES:
    p = os.path.join(EMB, f'{n}_train.npy')
    if os.path.exists(p):
        print('✅', n, np.load(p, mmap_mode='r').shape)
    else:
        print('❌ HILANG', n); ok = False
assert ok, "Ada embedding train hilang — selesaikan extract_all dulu"
print('\nSemua embedding train siap untuk grid')

---
## 🚀 JALANKAN GRID

In [ ]:
# Cell 3 — Jalankan probe_grid.py
# 26 run CV (5 backbone x 4 head + 3 concat x 2 head). ~10-30 menit di CPU.
# OOF tiap kombinasi + tabel probe_grid.csv tersimpan ke Drive (CFG.save_dir).
import os
os.chdir('/content/satria-data-bdcugm02/track_b/src')
!python ../experiments/probe_grid.py

---
## 📊 BACA HASIL

In [ ]:
# Cell 4 — Tampilkan tabel keputusan, urut mean, + tandai pemenang menurut aturan
import pandas as pd, sys
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/src')
from config import CFG

df = pd.read_csv(os.path.join(CFG.save_dir, "probe_grid.csv"))
df = df.sort_values("mean", ascending=False).reset_index(drop=True)

print("=== SEMUA KOMBINASI (urut mean) ===")
print(df[["combo","head","mean","min","std"]].to_string(index=False))

# Aturan seleksi: kandidat dalam 0.002 dari mean tertinggi, pilih min tertinggi
top_mean = df["mean"].iloc[0]
close = df[df["mean"] >= top_mean - 0.002].copy()
winner = close.sort_values("min", ascending=False).iloc[0]

print("\n=== KANDIDAT TERATAS (dalam 0.002 dari puncak) ===")
print(close[["combo","head","mean","min","std"]].to_string(index=False))
print(f"\n PEMENANG (mean tinggi + min terbaik + stabil):")
print(f"   {winner['combo']} / {winner['head']}")
print(f"   mean={winner['mean']:.4f}  min={winner['min']:.4f}  std={winner['std']:.4f}")

In [ ]:
# Cell 5 — Cek: apakah DINOv3 / concat benar menambah di atas SigLIP tunggal terbaik?
best_single_siglip = df[df["combo"].str.startswith("siglip") & ~df["combo"].str.contains(r"\+")]
best_siglip = best_single_siglip.sort_values("mean", ascending=False).iloc[0]
best_dino = df[df["combo"].str.startswith("dinov3") & ~df["combo"].str.contains(r"\+")].sort_values("mean", ascending=False).iloc[0]
best_concat = df[df["combo"].str.contains(r"\+")].sort_values("mean", ascending=False).iloc[0]

print(f"SigLIP tunggal terbaik : {best_siglip['combo']}/{best_siglip['head']}  mean={best_siglip['mean']:.4f} min={best_siglip['min']:.4f}")
print(f"DINOv3 tunggal terbaik : {best_dino['combo']}/{best_dino['head']}  mean={best_dino['mean']:.4f} min={best_dino['min']:.4f}")
print(f"Concat terbaik         : {best_concat['combo']}/{best_concat['head']}  mean={best_concat['mean']:.4f} min={best_concat['min']:.4f}")
print()
gain = best_concat["mean"] - best_siglip["mean"]
if gain > 0.001 and best_concat["min"] >= best_siglip["min"]:
    print(f"✅ Concat menambah nyata (+{gain:.4f}, min tidak turun) — DINOv3 layak dipakai di ensemble")
elif gain > 0.001:
    print(f"⚠️ Concat mean naik +{gain:.4f} TAPI min turun — hati-hati, mungkin overfit OOF")
else:
    print(f"➖ Concat tidak menambah (Δ={gain:+.4f}) — SigLIP tunggal mungkin sudah cukup")

---
## Langkah berikutnya

1. **Catat pemenang** dari Cell 4 — itu komposisi yang diserahkan ke Track C.
2. **OOF pemenang** (`oof_<combo>_<head>.npy` di Drive) → Track C untuk threshold tuning + weight search.
3. **Belum uji TTA** — grid ini pakai non-TTA. Kalau mau, uji varian `_tta` terpisah:
   ganti `load_embeddings(name, "train")` → `load_embeddings(name+"_tta", "train")` dan
   bandingkan. Lakukan HANYA untuk pemenang, jangan seluruh grid (hemat waktu + hindari overfit).
4. **Safety net** kalau masih 0/3 — CV di sini sudah cukup untuk submission pertama.